## Setup

In [1]:
import sys 
import os
import IPython


# Env. vars to check if the notebook is running on colab, kaggle etc. 
IS_COLAB = "google.colab" in sys.modules 
IS_KAGGLE = "kaggle_secrets" in sys.modules 
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

if IS_COLAB:
    # Install the packages 
    %pip install -q -U tensorflow-addons
    %pip install -q -U transformers
    %pip install -q -U datasets
    print("You can safely ignore the package incompatibility errors.")
    # Mount the drive 
    from google.colab import drive 
    drive.mount("/drive")

# NOTE: This needs to be done for correct notebook behavior with the transformers 
# library 
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm 
from copy import deepcopy 

import random 
import shutil 
# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

# Pytorch imports 
import torch
import torch.nn as nn 
from torch.utils.data import Dataset, DataLoader

from typing import List, Tuple

# To plot pretty figures
import seaborn as sns
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

sns.set_theme()

In [3]:
# Project Paths
NOTEBOOK_NAME = "2.0-Experiments"
PROJECT_ROOT_DIR = "."

# --- Result dirs. 
OUTPUT_DIR = os.path.join(PROJECT_ROOT_DIR,"outputs",NOTEBOOK_NAME)
os.makedirs(OUTPUT_DIR,exist_ok=True)


LOGGING_DIR = os.path.join(OUTPUT_DIR,"logs")
os.makedirs(LOGGING_DIR,exist_ok=True)

In [4]:
# Setup logging for this notebook 
from LLMDialogue.utils.logging import configure_logger, create_file_handler, create_stream_handler
from LLMDialogue.utils.general import get_system_device
import logging

configure_logger(
    handlers=[
        create_stream_handler(logging_level=logging.WARNING), 
        create_file_handler(
            outdir=LOGGING_DIR, 
            logging_level=logging.DEBUG, 
            reset_outdir=False # Keep logs across runs. 
        )
    ]
)


In [5]:

GLOBAL_DEVICE = get_system_device()
# Shared env. vars. 
GLOBAL_SEED = 42 
RESET_OUTDIR = True
# Configuring env.
if  GLOBAL_SEED is not None:
    # to make this notebook's output stable across runs
    np.random.seed(GLOBAL_SEED) 
    torch.manual_seed(GLOBAL_SEED)
# NOTE: Should be mps for mac. 
GLOBAL_DEVICE

device(type='mps')

In [6]:
#Reset all but the logs dir in the output folder
from LLMDialogue.utils.general import remove_dir, get_subdirs_in_dir
if RESET_OUTDIR:
    subdirs = get_subdirs_in_dir(OUTPUT_DIR)
    for subdir in subdirs:
        if not subdir in ['logs']:
            remove_dir(subdir)


## Experiments

In [7]:
from LLMDialogue.pipelines import CPPipelineConfig, CPPipeline
from LLMDialogue.models import GPT2, TurnGPT
from LLMDialogue.dm.ICCdm import ICCHFLMDM, ICCSpokenLanguageModelingDM

In [8]:
EXPERIMENT_DIR = f"{OUTPUT_DIR}/experiments"
os.makedirs(EXPERIMENT_DIR,exist_ok=True)

### Data Modules

There are various data modules, each with their own configurations that 
we need to be able to run. 

#### In Conversation Corpus

In [9]:
from LLMDialogue.dm.ICCdm import ICCHFLMDM, ICCSpokenLanguageModelingDM

In [10]:
DM_SAVE_DIR = f"{EXPERIMENT_DIR}/dm"
os.makedirs(DM_SAVE_DIR,exist_ok=True)

In [11]:
# NOTE: Each dataset has both different variants (labels and no labels) 
# as well as different underlying data dirs. 
# For GPT-2 fine-tuning, we want to use ICCHFLMDM with variant=labels 
# For TurnGPT fine-tuning, we want to use ICCSpokenLanguageModelingDM with 
# variant=no_Labels

# NOTE: The data modules cannot be instantiated until we have specified the 
# relevant model. 

DSET_VARIANTS = ("labels", "no_labels")


# NOTE: Dset variant, tokenizer, and save_dir need to be specified later. 
ICCHFLMDM_CONFIGS = {
    "data_block_size" : 128,
    "batched" : True,
}

# NOTE: Dset variant, tokenizer, and save_dir need to be specified later. 
ICCSpokenLanguageModelingDM_CONFIGS = {
    "batch_size" : 16, 
    "max_length" : 1024, 
    "chunk_size" : 128, 
    "num_workers" : 1, 
    "pin_memory" : True, 
    "num_proc" : None, 
    "use_cache" : False, 
    "batched" : True,
}



### Models 

We currently want to be able to load the GPT2 and TurnGPT models

#### GPT-2

In [12]:
from LLMDialogue.models import GPT2

In [13]:
_GPT2_MODEL_CONFIGS = {
    "model_checkpoint" : "gpt2", 
    "tokenizer_checkpoint" : "gpt2", 
    "tokenizer_pad_token" : "<PAD>", 
    "tokenizer_eos_token" : "<|endoftext|>",
    "tokenizer_additional_special_tokens" : [
        "<SP1>", "<SP2>", "<START>", "<END>"
    ]
}



In [14]:
model = GPT2(GLOBAL_SEED)
model.load(**_GPT2_MODEL_CONFIGS)

#### TurnGPT 

In [15]:
from LLMDialogue.models import TurnGPT

In [16]:
_TURNGPT_MODEL_CONFIG = {
    "pretrained_model_name_or_path" : "gpt2", 
    "model_head" : "LMHead"
}



In [17]:
model = TurnGPT(GLOBAL_SEED)
model.load(**_TURNGPT_MODEL_CONFIG)

/opt/miniconda3/envs/gpt/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Additional special tokens map:
	bos_token: <|endoftext|>
	eos_token: <ts>
	unk_token: <|endoftext|>
	pad_token: <|endoftext|>
	additional_special_tokens: ['<SP1>', '<SP2>']

NOTE: This tokenizer assumes that there are 2 speakers
Initialized special wte for <ts> to average of ['!', '?', '.']
This will make the <ts> semantically closer to ['!', '?', '.']


### ML Tracker 



In [18]:
# We need to define the configurations for the WANDB tracker 
# NOTE: We need to make sure we add the logging dir to be the 
# same as the config dir. 
_WANDB_CONFIGS = {
    "project" : "gpt_monologue_to_dialogue", 
    "entity" : "hilab", 
    "mode" : "online", 
    "run_name" : "test", 
    "log_model" : "end",
}



### Fine-tuning

We would like to fine-tune various models (GPT2, TurnGPT etc.) with various 
fine-tuning datasets. There are fine-tuning variations based both on the 
dataset as well as the data directories (we have two datasets based on different 
ICC files). 


In [19]:
from LLMDialogue.pipelines import (   
    LLMFinetunePipeline, 
    LLMFinetunePipelineConfig, 
)
from LLMDialogue.pipelines.utils import run_pipeline_with_tracker

In [20]:
# TODO: There is variation in both the variant of the datasets and the 
# data directories that are used. 

FINETUNE_SAVE_DIR = f"{EXPERIMENT_DIR}/finetune"
os.makedirs(FINETUNE_SAVE_DIR,exist_ok=True)

In [21]:
_DATA_FILE_PATHS = {
    "28_train_14_test" : {
        "train" : "/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/data/raw/ICC/28_train_14_test_set/train", 
        "test" : "/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/data/raw/ICC/28_train_14_test_set/test"
    }, 
    "5_train_37_test" : {
        "train" : "/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/data/raw/ICC/5_train_37_test_set/train", 
        "test" : "/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/data/raw/ICC/5_train_37_test_set/test"
    }
}

In [22]:
# NOTE: We need to specify which data files we want to use!!!

_DATA_FILE_VARIANT = "5_train_37_test"

#### GPT-2 using ICC with various data files

In [23]:
save_dir = f"{FINETUNE_SAVE_DIR}/gpt2/{_DATA_FILE_VARIANT}"
os.makedirs(save_dir,exist_ok=True)

In [24]:
model = GPT2(GLOBAL_SEED)
model.load(**_GPT2_MODEL_CONFIGS)

In [25]:
# First, we need to update the args for the data module and load it. 

_DM_CONFIGS = deepcopy(ICCHFLMDM_CONFIGS)
_DM_CONFIGS.update({
    "dset_variant" : "labels",
    "tokenizer" : model.tokenizer, 
    "save_dir" : save_dir, 
    "data_files" : _DATA_FILE_PATHS[_DATA_FILE_VARIANT]
})

dm = ICCHFLMDM(**_DM_CONFIGS)


In [26]:
# Create the pipeline 

# NOTE: The dm and save dir are defined by the pipeline. 
_GPT2_FINETUNE_CONFIGS = {
    "num_train_epochs" : 1, 
    "per_device_train_batch_size" : 8,
    "per_device_eval_batch_size" : 8,
    "warmup_steps" : 300,
    "run_name": None,
}

pipe_config = LLMFinetunePipelineConfig(
    model=model, 
    dm=dm, 
    report_to="none", 
    finetune_kwargs=_GPT2_FINETUNE_CONFIGS
)
finetune_pipe = LLMFinetunePipeline(pipe_config)



In [27]:
# Update the kwargs for the WANDB tracker 
_TRACKER_CONFIGS = deepcopy(_WANDB_CONFIGS)
_TRACKER_CONFIGS.update({
    "logging_dir" : pipe_config.cache_dir
})


In [28]:
# Now, start the finetuning process and log with wandb. 
# tracker = run_pipeline_with_tracker(
#     tracker_type="wandb", 
#     pipe=finetune_pipe, 
#     **_TRACKER_CONFIGS
# )
# tracker

#### TurnGPT with various data files

In [29]:
save_dir = f"{FINETUNE_SAVE_DIR}/turngpt/{_DATA_FILE_VARIANT}"
os.makedirs(save_dir,exist_ok=True)

In [30]:
model = TurnGPT(GLOBAL_SEED)
model.load(**_TURNGPT_MODEL_CONFIG)

/opt/miniconda3/envs/gpt/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Additional special tokens map:
	bos_token: <|endoftext|>
	eos_token: <ts>
	unk_token: <|endoftext|>
	pad_token: <|endoftext|>
	additional_special_tokens: ['<SP1>', '<SP2>']

NOTE: This tokenizer assumes that there are 2 speakers
Initialized special wte for <ts> to average of ['!', '?', '.']
This will make the <ts> semantically closer to ['!', '?', '.']


In [31]:
_DM_CONFIGS = deepcopy(ICCSpokenLanguageModelingDM_CONFIGS)
_DM_CONFIGS.update({
    "dset_variant" : "labels",
    "tokenizer" : model.tokenizer, 
    "save_dir" : save_dir, 
    "data_files" : _DATA_FILE_PATHS[_DATA_FILE_VARIANT]
})

dm = ICCSpokenLanguageModelingDM(**_DM_CONFIGS)


Map:   0%|          | 0/3901 [00:00<?, ? examples/s]

Map:   0%|          | 0/995 [00:00<?, ? examples/s]

Map:   0%|          | 0/833 [00:00<?, ? examples/s]

Map:   0%|          | 0/765 [00:00<?, ? examples/s]

Map:   0%|          | 0/615 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/709 [00:00<?, ? examples/s]

Map:   0%|          | 0/485 [00:00<?, ? examples/s]

Map:   0%|          | 0/744 [00:00<?, ? examples/s]

Map:   0%|          | 0/715 [00:00<?, ? examples/s]

Map:   0%|          | 0/572 [00:00<?, ? examples/s]

In [32]:
# NOTE: The dm and save dir are defined by the pipeline. 


_TURNGPT_FINETUNE_CONFIGS = {
    "max_epochs" : 1, 
    "log_every_n_steps" : 1, 
    "report_to" : "wandb"
}

pipe_config = LLMFinetunePipelineConfig(
    model=model, 
    dm=dm, 
    report_to="none", 
    finetune_kwargs=_TURNGPT_FINETUNE_CONFIGS
)
finetune_pipe = LLMFinetunePipeline(pipe_config)

In [33]:
# Update the kwargs for the WANDB tracker 
_TRACKER_CONFIGS = deepcopy(_WANDB_CONFIGS)
_TRACKER_CONFIGS.update({
    "logging_dir" : pipe_config.cache_dir
})


In [34]:
# # Now, start the finetuning process and log with wandb. 
# tracker = run_pipeline_with_tracker(
#     tracker_type="wandb", 
#     pipe=finetune_pipe, 
#     **_TRACKER_CONFIGS
# )
# tracker

### Conditional Probabilities for the Speaker Identity Dataset

In [35]:
from LLMDialogue.pipelines import CPPipeline, CPPipelineConfig
from LLMDialogue.datasets.speakerIdentity import SpeakerIdentityDataset, objectify
from LLMDialogue.models import LanguageModel

In [36]:
CP_DIR = f"{EXPERIMENT_DIR}/CP"
os.makedirs(CP_DIR,exist_ok=True)

In [37]:
def predict_speaker_identity_dset(
    model : LanguageModel,
    dset : SpeakerIdentityDataset,
    save_dir : str,
    N : int =  -1, 
    context_buffer_size : int = 512, 
) -> Tuple:
    splits = objectify(dset)
    test_df = splits.data_df_map['test']
    # This is a list of all of the conversation IDs. 
    conv_ids = list(set(test_df['conversationID'].tolist()))
    results = list()
    for conv_id in conv_ids:
        # Get the utterances for this conversation. 
        utterances = test_df[test_df['conversationID'] == conv_id]['utterance'].tolist()
        # Set up the pipeline. 
        pipe_config = CPPipelineConfig(
            model = model,
            utterances=utterances,
            N=N, 
            context_buffer_size=context_buffer_size
        )
        pipe = CPPipeline(pipe_config)
        pipe.run()
        res = pipe.info().data
        # Create a dataframe from the results and save. 
        res_df = pd.DataFrame(res)
        results.append(res_df.copy())
    # Concat the df into a larger one and save
    df = pd.concat(results)
    save_fpath = f"{save_dir}/CP.csv"
    df.to_csv(save_fpath)
    return df, save_fpath


#### GPT-2

In [38]:
from LLMDialogue.datasets.speakerIdentity import (
    SpeakerIdentityDataset, 
    SpeakerIdentityDatasetConfig, 
    SpeakerIdentityDataSplits, 
    objectify
)
from LLMDialogue.datasets import load

In [39]:
_DSET_KWARGS = {
    "dset_name" : "speakerIdentity", 
    "config_name" : "labels"
}

In [40]:
save_dir = f"{CP_DIR}/gpt2/{_DSET_KWARGS['config_name']}"
os.makedirs(save_dir,exist_ok=True)

In [41]:
# First, load the right model from the right checkpoint. 
# TODO: Update the checkpoint path before loading!!!!
model = GPT2(GLOBAL_SEED)
model.load(**_GPT2_MODEL_CONFIGS)

In [42]:
# Load the dataset. 
# NOTE: Fot GPT-2, we want to use the labels dataset. 
dset = load(
    **_DSET_KWARGS
)
print(dset)


{
    "date_created": "23_05_2024__13_16_41",
    "dset_name": "speakerIdentity",
    "variant_name": "labels",
    "version": "0.0.1",
    "data_dir": "/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/data/raw/speaker_identity_stims",
    "data_files": null,
    "force_regenerate": false,
    "config_id": "3f884a446b7accfb4f48c4b3955de63a670afb2b7c1eac6a34f84bb795bb9522",
    "cache_dir": "/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/cache/datasets/3f884a446b7accfb4f48c4b3955de63a670afb2b7c1eac6a34f84bb795bb9522",
    "archive_dir": "/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/cache/datasets/3f884a446b7accfb4f48c4b3955de63a670afb2b7c1eac6a34f84bb795bb9522/archive",
    "compress_archive": true,
    "seed": 42,
    "verbose": false,
    "additional_special_tokens": {
        "sp_start_token": "<START>",
        "sp_end_token": "<END>"
    }
}


In [43]:
df, save_fpath = predict_speaker_identity_dset(
    model=model, 
    dset=dset, 
    save_dir=save_dir,
    N=-1, 
    context_buffer_size=512
)

Processing turns: 100%|██████████| 4/4 [00:00<00:00, 14.70it/s]


In [44]:
df 

,turn_no,word_no,context,word,last_word_prob
0,1,0,<START>,<SP1>,1.895725e-18
1,1,1,<START> <SP1>,i,0.000000e+00
2,1,2,<START> <SP1> i,haven't,0.000000e+00
3,1,3,<START> <SP1> i haven't,seen,0.000000e+00
4,1,4,<START> <SP1> i haven't seen,the,0.000000e+00
...,...,...,...,...,...
12,2,0,<START> <SP1> it's great that you could cover ...,<SP1>,0.000000e+00
13,2,1,<START> <SP1> it's great that you could cover ...,for,0.000000e+00
14,2,2,<START> <SP1> it's great that you could cover ...,real,0.000000e+00
15,2,3,<START> <SP1> it's great that you could cover ...,<SP1>,0.000000e+00
